In [ ]:
!pip install -q fairseq sacrebleu sentencepiece

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
Requested omegaconf<2.1 from https://files.pythonhosted.org/packages/d0/eb/9d63ce09dd8aa85767c65668d5414958ea29648a0eec80a4a7d311ec2684/omegaconf-2.0.6-py3-none-any.whl (from fairseq) has invalid metadata: .* suffix can only be used with `==` or `!=` operators
    PyYAML (>=5.1.*)
            ~~~~~~^
Please use pip<24.1 if you need to use this version.
Requested omegaconf<2.1 from https://files.pythonhosted.org/packages/e5/f6/043b6d255dd6fbf2025110cea35b87f4c5100a181681d8eab496269f0d5b/omegaconf-2.0.5-py3-none-any.whl (from fairseq) has invalid metadata: .* suffix can only be used with `==` or `!=` operators
    PyYAML (>=5.1.*)
            ~~~~~~^
Please use pip<24.1 if you need to use this version.
Requested omegaconf<2.1 from https://files.pythonhosted.org/packages/92/b1/4f3023143436f12c98bab53f0b3db617bd18a

In [ ]:
!pip install indic-nlp-library

In [ ]:
!nvidia-smi

Sat Feb 21 12:20:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
from indicnlp import common, loader
import pandas as pd

In [ ]:
factory=IndicNormalizerFactory()
normalizer=factory.get_normalizer("ne")

In [ ]:
text_columns = ['MAGAR', 'NEPALI']

df = pd.read_csv('magar-nepali.csv')

for col in text_columns:
  df[col] = df[col].apply(lambda x: normalizer.normalize(str(x)) if pd.notna(x) else x)

df.to_csv('output-normalized.csv', index=False)

In [ ]:
print("Total sentence pairs in the dataset: ", len(df))

Total sentence pairs in the dataset:  4298


# Start Here next time

In [ ]:
# Shuffle data and split into train/valid/test sets

import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('output-normalized.csv')

# shuffle
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Split: 80/10/10
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
valid_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Save
# train_df.to_csv('train.mgr-np.csv', index=False)
# valid_df.to_csv('valid.mgr-np.csv', index=False)
# test_df.to_csv('test.mgr-np.csv', index=False)

print(f"Train: {len(train_df)} | Valid: {len(valid_df)} | Test: {len(test_df)}")

Train: 3438 | Valid: 430 | Test: 430


In [ ]:
# write both column in a single text file
with open('all-text.txt', 'w', encoding="utf-8") as f:
  for sent in pd.concat([df['MAGAR'], df['NEPALI']]):
    f.write(str(sent) + '\n')

In [ ]:
import sentencepiece as spm
# Train Sentencepiece
spm.SentencePieceTrainer.train(
    input="all-text.txt",
    model_prefix="spm-twelve-hundred",
    vocab_size=1200,
    character_coverage=1.0,
    model_type="bpe",
    user_defined_symbols=['<2np>', '<2mgr>'],
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3
)

In [ ]:
# Load the model
sp = spm.SentencePieceProcessor()
sp.load("spm-twelve-hundred.model") # the name provided in model_prefix, change vocab_size for better adjustment like 4000, 2000, etc


True

In [ ]:
# Convert sentence into space-separated BPE subword
def encode_bpe(text):
  return " ".join(sp.encode(str(text), out_type=str))

In [ ]:
import os
os.makedirs("data", exist_ok=True)

In [ ]:
def write_files(src_lines, tgt_lines, src_path, tgt_path):
  with open(src_path, 'w', encoding="utf-8") as f_src, open(tgt_path, 'w', encoding="utf-8") as f_tgt:
    f_src.write("\n".join(src_lines) + "\n")
    f_tgt.write("\n".join(tgt_lines) + "\n")

In [ ]:
# Encoding Nepali -> Magar pairs
np2mgr_src = train_df['NEPALI'].apply(lambda x: encode_bpe("<2mgr>" + x))
np2mgr_tgt = train_df['MAGAR'].apply(encode_bpe)

In [ ]:
# Encode Magar -> Nepali pairs
mgr2np_src = train_df['MAGAR'].apply(lambda x: encode_bpe("<2np>" + x))
mgr2np_tgt = train_df['NEPALI'].apply(encode_bpe)

In [ ]:
# Stact both directions together
train_src = pd.concat([np2mgr_src, mgr2np_src]).tolist()
train_tgt = pd.concat([np2mgr_tgt, mgr2np_tgt]).tolist()

In [ ]:
write_files(train_src, train_tgt, "data/train.np", "data/train.mgr")

In [ ]:
# Validation and testing only in one direction since we neet to monitor and evaluating one direction is enough
write_files(
    valid_df['NEPALI'].apply(lambda x: encode_bpe("<2mgr>" + x)).tolist(),
    valid_df['MAGAR'].apply(encode_bpe).tolist(),
    "data/valid.np", "data/valid.mgr"
            )

In [ ]:
write_files(
    test_df['NEPALI'].apply(lambda x: encode_bpe("<2mgr>" + x)).tolist(),
    test_df['MAGAR'].apply(encode_bpe).tolist(),
    "data/test.np", "data/test.mgr"
            )

NOTE:

*   train.np can actually contain both  Nepali and Magar text (with tags) in source
*   The model learns to handle this via the tags



In [ ]:
# Convert SentencePiece vocab to Fairseq format
with open("spm-eight-hundred.vocab", 'r', encoding="utf-8") as fin, open("fairseq_dict.txt", 'w', encoding="utf-8") as fout:
  for line in fin:
    token = line.strip().split("\t")[0]
    if token in ['<unk>', '<s>', '</s>', '<pad>']:
      continue
    fout.write(f"{token} 1\n")

In [ ]:
!pip install git+https://github.com/One-sixth/fairseq.git

  Cloning https://github.com/One-sixth/fairseq.git to /tmp/pip-req-build-qyrpy8wq
  Running command git clone --filter=blob:none --quiet https://github.com/One-sixth/fairseq.git /tmp/pip-req-build-qyrpy8wq
  Resolved https://github.com/One-sixth/fairseq.git to commit 44800430a728c2216fd1cf1e8daa672f50dfacba
  Running command git submodule update --init --recursive -q
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached sacrebleu-2.6.0-py3-none-any.whl.metadata (39 kB)
  Using cached bitarray-3.8.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (34 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 340.3/340.3 kB 24.3 MB/s eta 0:00:00
  Created wheel for fairseq: filename=fairseq-0.12.3-cp312-c

In [ ]:
import fairseq
print(fairseq.__version__)

0.12.3


In [ ]:
# Binarize the dataset for Fairseq
# !python fairseq_cli.preprocess \
!fairseq-preprocess \
--source-lang np \
--target-lang mgr \
--trainpref data/train \
--validpref data/valid \
--testpref data/test \
--destdir data-bin \
--joined-dictionary \
--srcdict fairseq_dict.txt \
--workers 8

2026-02-21 14:31:47.899818: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771684307.921120    8754 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771684307.928359    8754 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771684307.946983    8754 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771684307.947019    8754 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771684307.947031    8754 computation_placer.cc:177] computation placer alr

In [ ]:
!pip install torch==2.4.0 --quiet
# Pytorch 2.6 changed torch.load default from weights_only=False to True for security reaons, and
# argparse.Namespace obj are not allowed under new strict mode.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
!fairseq-train data-bin \
--source-lang np --target-lang mgr \
--arch transformer_iwslt_de_en \
--share-decoder-input-output-embed \
--optimizer adam --adam-betas '(0.9, 0.98)' \
--clip-norm 1.0 \
--lr 5e-4 --lr-scheduler inverse_sqrt \
--warmup-updates 2000 \
--dropout 0.3 --weight-decay 0.0001 \
--criterion label_smoothed_cross_entropy \
--label-smoothing 0.1 \
--max-tokens 4000 \
--max-epoch 100 \
--patience 10 \
--save-dir checkpoints \
--eval-bleu \
--eval-bleu-args '{"beam": 5, "max_len_a": 1.2, "max_len_b": 10}' \
--eval-bleu-detok space \
--eval-bleu-remove-bpe sentencepiece \
--best-checkpoint-metric bleu \
--maximize-best-checkpoint-metric \
--fp16

2026-02-21 15:19:10.345781: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771687150.578089   21594 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771687150.647984   21594 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771687151.136596   21594 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771687151.136637   21594 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771687151.136642   21594 computation_placer.cc:177] computation placer alr

In [ ]:
!fairseq-generate data-bin --path checkpoints/checkpoint_best.pt \
--batch-size 64 --beam 5 --remove-bpe=sentencepiece \
--source-lang np --target-lang mgr --gen-subset test > test_output.txt

2026-02-21 15:51:14.556245: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771689074.577145   29830 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771689074.583370   29830 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771689074.599238   29830 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771689074.599265   29830 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771689074.599269   29830 computation_placer.cc:177] computation placer alr

In [ ]:
# Compute BLEU score on the test set
import sacrebleu
# Read the output file and collect hypotheses and references
hyps = [""] * len(test_df) # placeholder list for hypotheses
with open("test_output.txt", "r", encoding="utf-8") as f:
  for line in f:
    if line.startswith("D-"): # Detokenized hypothesis line
# Format: D-idx \t score \t translation
      parts = line.strip().split("\t")
      idx = int(parts[0][2:]) # get the index after "D-"
      translation = parts[2]
      hyps[idx] = translation
# Get reference sentences in original form (Magar)
refs = [str(text) for text in test_df['MAGAR']]
# Calculate BLEU
bleu = sacrebleu.corpus_bleu(hyps, [refs])
print(f"BLEU score on test set: {bleu.score:.2f}")

BLEU score on test set: 33.35
